# Chapter 0 — Mathematical and Neuroscience Refreshers

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*

**Purpose of this chapter.**  
The remainder of this book assumes a working command of elementary linear algebra, probability, and the coarsest facts of systems neuroscience. Chapter 0 does not replace a full course in any of those subjects. It gathers, in one place, the minimum that a student from computer science, electrical engineering, or the life sciences will need in order to read Chapters 1–12 without stalling on notation.

Work through the code cells. They are part of the text, not optional illustrations.

**Learning outcomes.** After this chapter you should be able to:

1. Compute inner products, norms, and a sample covariance matrix, and state what an eigenvector of that matrix represents.
2. Apply Bayes’ rule to a two-class decoding problem with explicit priors and likelihoods.
3. Explain why a random shuffle of time-series windows leaks information across the training and test sets.
4. Describe a neuron, a synapse, and a population rhythm at the level required to interpret EEG and implanted recordings.
5. State, in one paragraph, why a scalp electrode and an intracortical electrode report different pictures of the same brain.

**How to use this notebook.**  
Run the cells in order. Exercises appear at the end of each part. Suggested solutions are not included in the public notebook; course instructors may release them separately.


---

## 0.1 Software conventions used in this book

All numerical examples use Python 3 with NumPy. Plotting uses Matplotlib. Later laboratory chapters add MNE-Python for electroencephalography; it is not required here.

Vectors are column-oriented in the prose ($x \in \mathbb{R}^n$) and one-dimensional `ndarray` objects in code, unless a matrix layout is required. Random seeds are fixed so that results in the text match a clean execution of the notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print("NumPy", np.__version__)


---

# Part A — Mathematical refresher

## 0.2 Vectors, inner products, and norms

A **vector** $x \in \mathbb{R}^n$ is an ordered list of $n$ real numbers. In this book a vector is typically a snapshot of a neural recording: $n$ electrodes at one time point, or $n$ time samples from one electrode, or $n$ features extracted from a trial.

The **inner product** of $x, y \in \mathbb{R}^n$ is

$$
\langle x, y \rangle = x^\top y = \sum_{i=1}^{n} x_i y_i.
$$

It measures alignment. If $\langle x, y \rangle$ is large and positive, the two patterns rise and fall together. If it is large and negative, they move in opposition. If it is near zero, they are orthogonal in this geometry.

The **Euclidean norm** is

$$
\|x\|_2 = \sqrt{\langle x, x \rangle}.
$$

Dividing a vector by its norm produces a unit vector. Many spatial filters in later chapters are most easily understood as directions on the unit sphere: they answer the question *along which combination of electrodes should we look?*


In [ ]:
x = np.array([1.0, 2.0, 2.0])
y = np.array([2.0, 1.0, 0.0])

inner = np.dot(x, y)
norm_x = np.linalg.norm(x)
cos_angle = inner / (norm_x * np.linalg.norm(y))

print(f"inner product <x, y> = {inner}")
print(f"||x||_2              = {norm_x:.4f}")
print(f"cosine of angle      = {cos_angle:.4f}")


A linear **spatial filter** $w \in \mathbb{R}^C$ applied to an instantaneous electrode vector $v_t \in \mathbb{R}^C$ is exactly an inner product:

$$
s_t = \langle w, v_t \rangle = w^\top v_t.
$$

Choosing $w$ is the design problem of Chapter 6. The algebra is already visible here: filtering is projection.


## 0.3 Matrices, covariance, and correlation

A recording over $C$ channels and $T$ time samples is stored as a matrix $X \in \mathbb{R}^{C \times T}$ (channels $\times$ time), or sometimes as its transpose. Always state the convention. This book uses **channels as rows** unless a library (for example MNE) dictates otherwise.

The **sample mean** across time is the vector

$$
\mu = \frac{1}{T} \sum_{t=1}^{T} x_t \in \mathbb{R}^{C}.
$$

The **sample covariance** is

$$
\Sigma = \frac{1}{T-1} \sum_{t=1}^{T} (x_t - \mu)(x_t - \mu)^\top \in \mathbb{R}^{C \times C}.
$$

$\Sigma_{ij}$ is large and positive when channels $i$ and $j$ co-vary. Covariance is the central second-order object of EEG analysis. Common spatial patterns, PCA, and several Riemannian methods in Chapter 7 all begin with $\Sigma$.


In [ ]:
C, T = 4, 200
# Four weakly structured channels: two pairs that co-vary.
source_a = np.random.randn(T)
source_b = np.random.randn(T)
X = np.vstack([
    source_a + 0.2 * np.random.randn(T),
    source_a + 0.2 * np.random.randn(T),
    source_b + 0.2 * np.random.randn(T),
    source_b + 0.2 * np.random.randn(T),
])

mu = X.mean(axis=1, keepdims=True)
X_centered = X - mu
Sigma = (X_centered @ X_centered.T) / (T - 1)

print("mean (per channel):")
print(np.round(mu.ravel(), 3))
print("\ncovariance matrix:")
print(np.round(Sigma, 3))


The diagonal of $\Sigma$ contains variances. The off-diagonal pattern above should show two blocks: channels 0–1 co-vary, channels 2–3 co-vary, and the cross-block entries are near zero. That block structure is a miniature of what a spatial filter tries to discover in real EEG, where the “sources” are cortical generators rather than the synthetic signals used here.

**Correlation** rescales covariance by the channel standard deviations so that each entry lies in $[-1, 1]$. Correlation is useful for display. Covariance is the object that most algorithms consume.


In [ ]:
std = np.sqrt(np.diag(Sigma))
corr = Sigma / np.outer(std, std)
print("correlation matrix:")
print(np.round(corr, 3))


## 0.4 Eigenvectors, eigenvalues, and principal component analysis

A non-zero vector $u$ is an **eigenvector** of a square matrix $A$ with **eigenvalue** $\lambda$ when

$$
A u = \lambda u.
$$

If $A$ is a covariance matrix, it is symmetric and positive semi-definite. Its eigenvalues are real and non-negative. Its eigenvectors can be chosen orthonormal.

**Principal component analysis (PCA)** is the eigen-decomposition of $\Sigma$:

$$
\Sigma = U \Lambda U^\top,
$$

where the columns of $U$ are principal axes and the diagonal of $\Lambda$ contains variances captured along those axes. The first principal axis is the direction in electrode space along which the recording fluctuates most.

PCA is not a decoder. It is a rotation that can compress $C$ noisy channels into $k < C$ coordinates, or that can help you see whether a recording is dominated by one artefact direction (often ocular) before you attempt to classify trials.


In [ ]:
eigvals, eigvecs = np.linalg.eigh(Sigma)
# eigh returns ascending eigenvalues; reverse for PCA convention
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

explained = eigvals / eigvals.sum()
print("eigenvalues:          ", np.round(eigvals, 3))
print("explained variance:   ", np.round(explained, 3))
print("first principal axis: ", np.round(eigvecs[:, 0], 3))


The first two eigenvalues should dominate, matching the two synthetic sources. The first principal axis should place similar weights on the two channels that share `source_a` (or `source_b`), up to sign. Sign is arbitrary in PCA: $u$ and $-u$ are equivalent.

The same geometry reappears under other names. A **common spatial pattern** (Chapter 6) is an eigenvector of a *pair* of covariance matrices, one per class. If you understand $\Sigma u = \lambda u$, you are one definition away from CSP.


## 0.5 A linear decoder as geometry

Suppose each trial yields a feature vector $z \in \mathbb{R}^d$ and a class label $y \in \{+1, -1\}$ (for example motor imagery of the left hand versus the right hand). A **linear classifier** predicts

$$
\hat{y} = \mathrm{sign}(w^\top z + b).
$$

The weight vector $w$ is a direction in feature space. Training chooses $w$ and $b$. Linear discriminant analysis, logistic regression, and a linear support-vector machine differ in how they choose $w$; they agree that the decision is a hyperplane.

This matters for BCI because many robust systems remain linear after a careful feature step. Non-linear models in Chapter 7 are justified when the feature map is incomplete, not because linearity is obsolete.


In [ ]:
# Two Gaussian classes in 2-D, linearly separable in expectation.
N = 80
z_neg = np.random.randn(N, 2) + np.array([-1.5, 0.0])
z_pos = np.random.randn(N, 2) + np.array([+1.5, 0.0])
Z = np.vstack([z_neg, z_pos])
y = np.hstack([-np.ones(N), np.ones(N)])

# Fisher-style closed form: w proportional to Sw^{-1}(m+ - m-)
m_neg, m_pos = z_neg.mean(0), z_pos.mean(0)
Sw = np.cov(z_neg, rowvar=False) + np.cov(z_pos, rowvar=False)
w = np.linalg.solve(Sw, m_pos - m_neg)
b = -0.5 * w @ (m_pos + m_neg)
y_hat = np.sign(Z @ w + b)
acc = np.mean(y_hat == y)
print(f"linear discriminant training accuracy = {acc:.3f}")
print(f"w = {np.round(w, 3)},  b = {b:.3f}")

fig, ax = plt.subplots(figsize=(5.2, 4.2))
ax.scatter(z_neg[:, 0], z_neg[:, 1], s=18, label="class −1", alpha=0.8)
ax.scatter(z_pos[:, 0], z_pos[:, 1], s=18, label="class +1", alpha=0.8)
# decision boundary w^T z + b = 0
xs = np.linspace(Z[:, 0].min() - 0.5, Z[:, 0].max() + 0.5, 50)
ys = -(w[0] * xs + b) / w[1]
ax.plot(xs, ys, color="black", linewidth=1.5, label="decision boundary")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("A linear decoder is a hyperplane")
ax.legend(frameon=True, loc="best")
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()


## 0.6 Probability, likelihood, and Bayes’ rule

Decoding is an inference problem. Let $Y$ be the unobserved class (or intended character, or intended phoneme) and let $X$ be the observed features. **Bayes’ rule** states

$$
P(Y=y \mid X=x) = \frac{P(X=x \mid Y=y)\, P(Y=y)}{P(X=x)}.
$$

- $P(Y=y)$ is the **prior**. In a P300 speller the prior over characters is not uniform: language structure concentrates probability on some letters given the letters already chosen. Language models in speech BCI are priors of this kind.
- $P(X=x \mid Y=y)$ is the **likelihood**: how plausible the recording is if the user intended $y$.
- $P(Y=y \mid X=x)$ is the **posterior**. A minimum-error decoder picks $\arg\max_y P(Y=y \mid X=x)$.

The denominator $P(X=x)$ does not depend on $y$ and can be ignored when the task is only to choose a class.


In [ ]:
# Toy two-class problem with one scalar feature.
# Class 0: X ~ N(0, 1); class 1: X ~ N(2, 1); prior P(Y=1) = 0.3
from math import erf, sqrt

def gaussian_pdf(x, mean, var=1.0):
    return np.exp(-0.5 * (x - mean) ** 2 / var) / np.sqrt(2 * np.pi * var)

prior1 = 0.3
prior0 = 1.0 - prior1
x_obs = 1.2
like0 = gaussian_pdf(x_obs, 0.0)
like1 = gaussian_pdf(x_obs, 2.0)
post1 = (like1 * prior1) / (like0 * prior0 + like1 * prior1)
print(f"observed x = {x_obs}")
print(f"likelihood P(x | Y=0) = {like0:.4f}")
print(f"likelihood P(x | Y=1) = {like1:.4f}")
print(f"posterior  P(Y=1 | x) = {post1:.4f}")
print("decision:", "class 1" if post1 > 0.5 else "class 0")


Change the prior and the same observation can flip the decision. That is not a trick. It is the reason a speech decoder with a language model outperforms an otherwise identical acoustic-only decoder, and the reason a workload estimator should not treat “high workload” as equally likely in every task context.

**Expected value and variance** will appear whenever we discuss noise. If $X$ is a random variable,

$$
\mathbb{E}[X] = \int x\, p(x)\, dx, \qquad \mathrm{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2].
$$

Averaging independent trials reduces the variance of the mean as $1/N$, which is why event-related potentials are estimated by averaging many repetitions—and why a BCI that needs fifty repetitions to see a P300 is not yet a conversation.


## 0.7 Evaluation, leakage, and time-series splits

Offline accuracy is the most abused number in BCI. Two failures are common.

**Failure 1 — ignoring the baseline.** A balanced two-class problem has a chance level of $0.5$. A P300 matrix speller with $K$ characters has a chance level near $1/K$ per attempted character if the decoder is tested character-wise, but the right baseline depends on how trials are pooled. Always state the chance level beside the score.

**Failure 2 — leakage across time.** Neural recordings are temporally correlated. If you cut a continuous record into windows and then *randomly shuffle* windows into a training set and a test set, adjacent windows that share artefacts, slow drifts, or the same mental state will appear on both sides of the split. The test score then measures memorisation of a session, not generalisation to a future session.

The default split in this book is **causal in time**: train on earlier blocks, test on later blocks. Where subject-independent models are claimed, entire subjects must be held out.


In [ ]:
def make_drifting_series(T=400):
    '''Scalar feature with a slow drift and alternating class blocks.'''
    t = np.arange(T)
    drift = 0.006 * t
    # class alternates every 50 samples so both classes appear early and late
    y = ((t // 50) % 2).astype(int)
    x = drift + 0.9 * y + 0.35 * np.random.randn(T)
    return x, y


x, y = make_drifting_series()


def accuracy(y_true, y_hat):
    return np.mean(y_true == y_hat)


def fit_threshold_classifier(x_train, y_train):
    # Midpoint between class means — deliberately simple.
    m0 = x_train[y_train == 0].mean()
    m1 = x_train[y_train == 1].mean()
    thresh = 0.5 * (m0 + m1)
    return thresh


def predict(x, thresh):
    return (x > thresh).astype(int)


# Leaky protocol: random shuffle of samples.
idx = np.random.permutation(len(x))
cut = len(x) // 2
train, test = idx[:cut], idx[cut:]
thresh_leak = fit_threshold_classifier(x[train], y[train])
acc_leak = accuracy(y[test], predict(x[test], thresh_leak))

# Honest protocol: earlier half versus later half.
thresh_time = fit_threshold_classifier(x[:200], y[:200])
acc_time = accuracy(y[200:], predict(x[200:], thresh_time))

print(f"leaky random split accuracy     = {acc_leak:.3f}")
print(f"causal time split accuracy      = {acc_time:.3f}")
print("The leaky number is not a valid estimate of future performance.")

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(x, color="steelblue", linewidth=1.0, label="feature")
ax.axvline(200, color="black", linestyle="--", linewidth=1, label="causal train / test split")
ax.set_xlabel("time sample")
ax.set_ylabel("feature value")
ax.set_title("Slow drift across alternating class blocks: why shuffle splits leak")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


The leaky split usually looks better because training samples are interleaved with test samples along the drift. A deployed BCI does not receive the future interleaved with the past. Chapters 6 and 7 will treat a leakage-safe split as part of the method, not as an extra courtesy.

**Cross-validation** remains useful, but the folds must respect time and, where relevant, recording session. *Leave-one-session-out* is often more informative than *leave-one-trial-out*.


## Exercises — Part A

**A1.** Let $x = (1, 0, -1)$ and $y = (2, 2, 0)$. Compute $\langle x, y \rangle$, $\|x\|_2$, and the cosine of the angle between them. Confirm with code.

**A2.** Construct a $3 \times 3$ covariance matrix from 100 samples of a three-channel signal in which channels 1 and 2 are identical and channel 3 is independent unit Gaussian noise. What structure do you expect, and what do you observe?

**A3.** In the Bayes example of Section 0.6, find the value of the prior $P(Y=1)$ that makes the posterior exactly $1/2$ when $x = 1.2$. Interpret the result.

**A4.** Explain, in at most four sentences, why standard $k$-fold cross-validation on randomly shuffled EEG windows is an invalid estimate of within-session generalisation when the windows overlap.

**A5.** A communication BCI reports 92 percent “trial accuracy” on a two-class task with 400 trials drawn from the same five-minute recording, shuffled, and split 80/20. List two questions you would ask before believing that the system will work tomorrow.


---

# Part B — Neuroscience refresher

## 0.8 What this part will not do

Neuroscience is not a single subject. Cellular biophysics, systems physiology, cognitive psychology, and clinical neurology each have their own textbooks. This part states only those facts that later chapters rely on:

- electrical signalling in neurons is the source of almost every signal a BCI records;
- synapses change with use, which is both a resource (learning to use an interface) and a problem (the statistics of the recording drift);
- electrodes do not record “thoughts”; they record voltage fields produced by populations, at a spatial grain that depends on where the metal sits.

If a sentence in Chapters 2–12 requires more biology than this, the sentence will supply it.


## 0.9 The neuron as a signalling device

A **neuron** is a cell specialised for receiving, integrating, and emitting electrical and chemical signals. For engineering purposes three parts matter.

- **Dendrites** collect input from other neurons.
- The **soma** (cell body) integrates that input.
- The **axon** conducts an output pulse, the **action potential** or spike, to synapses on downstream cells.

An action potential is a brief, stereotyped all-or-none pulse, on the order of one millisecond. Its shape carries little information. Timing and rate do. A cortical neuron may fire a few times per second on average, with bursts far above that rate during relevant computation.

Two modelling remarks used later:

1. At the level of EEG, individual spikes are invisible. The scalp signal is a slow mixture of **postsynaptic potentials** in aligned populations, not a spike raster.
2. At the level of an intracortical microelectrode, both spikes (from one or a few nearby cells) and the **local field potential** (from a small surrounding volume) can be recorded. They answer different questions.


## 0.10 Synapses and plasticity

A **synapse** is a junction at which one neuron influences another, usually by releasing neurotransmitter that opens or modulates ion channels on the receiving cell. The effective strength of that influence is the **synaptic weight** in every neural-network metaphor you have already met.

**Synaptic plasticity** is the activity-dependent change of that weight. The names you will see again:

- **LTP / LTD** — long-term potentiation and depression, lasting strengthening or weakening;
- **STDP** — spike-timing-dependent plasticity, in which the order of pre- and postsynaptic spikes influences the sign of the change.

You do not need the molecular cascade. You need the systems consequence: the mapping from intent to recorded signal is not a fixed function. Users learn. Decoders must be allowed to adapt. A closed-loop rehabilitation protocol (Chapter 10) is an attempt to *use* plasticity rather than merely survive it.

Short-term facilitation and depression also exist, on the scale of milliseconds to seconds. They contribute to the non-stationarity that makes a morning calibration degrade by afternoon.


## 0.11 Populations, fields, and rhythms

A single pyramidal cell in cortex is a small battery sitting in a conductive volume. When thousands of such cells receive coordinated synaptic input, their postsynaptic currents add. The resulting extracellular voltage is a **field potential**.

- Close to the cells, a microelectrode sees a **local field potential** (LFP).
- On the cortical surface, an ECoG electrode sees a centimetre-scale mixture.
- On the scalp, an EEG electrode sees a still broader, skull-filtered mixture, of the order of several square centimetres of cortex, biased toward sources that are radially oriented and close to the electrode.

**Oscillations** are rhythmic fluctuations of these fields. Conventional bands are bookkeeping devices, not natural kinds:

| Band   | Approximate range | Often associated with (loosely)        |
|--------|-------------------|----------------------------------------|
| Delta  | 0.5–4 Hz          | deep sleep, some slow cognitive states |
| Theta  | 4–8 Hz            | memory, midline cognitive control      |
| Alpha  | 8–13 Hz           | idle posterior cortex, attentional gate|
| Beta   | 13–30 Hz          | motor maintenance, some effort states  |
| Gamma  | 30–80+ Hz         | local population engagement            |

Treat every “associated with” as a statistical tendency, not a code-book. Motor-imagery BCIs often rely on attenuation of mu/beta rhythms over sensorimotor cortex (event-related desynchronisation). That is a population phenomenon. It does not imply that the user has a unique “mu thought.”


## 0.12 Maps that an engineer may rely on

The cortex is organised, in part, as maps.

- **Primary motor cortex** (precentral gyrus) contains a coarse map of the body. Activity here correlates with intended movement and is the classical target of motor BCIs.
- **Somatosensory cortex** (postcentral gyrus) maps the body surface and is relevant if the interface must restore touch rather than only read intent.
- **Speech-related cortex** includes superior temporal areas for acoustic and phonetic processing and inferior frontal and ventral motor areas for articulation. Attempted-speech BCIs record from some of these regions.
- **Posterior parietal cortex** participates in planning and in visuospatial goals. Cognitive signals used in some implanted research—goals rather than pull-direction—often live here.
- **Prefrontal cortex** participates in control, working memory, and value. It is a tempting cognitive-BCI target and a difficult one: representations are distributed, mixed, and context-dependent.

Subcortical structures (basal ganglia, thalamus, hippocampus) matter enormously to disease and to deep-brain stimulation. Most introductory BCI recordings still come from cortex or from the scalp above it. Do not infer from that convenience that cognition happens only on the surface.


## 0.13 What a BCI electrode actually reports

The following table is the biological counterpart of the modality table in Chapter 3. Learn the qualitative grain, not the exact millimetre figures.

| Situation                         | Typical content of the recording                                      | What it is not                      |
|-----------------------------------|------------------------------------------------------------------------|-------------------------------------|
| Scalp EEG                         | Blurred, attenuated fields of large, aligned cortical populations       | A transcript of a single cell or a thought |
| Cortical surface (ECoG)           | Intermediate fields; better spatial grain than EEG, still a population | A complete map of a cognitive variable |
| Intracortical microelectrode      | Spikes of one or a few cells plus LFP                                  | A guaranteed stable unit for years  |
| Endovascular stent-mounted sensor | Fields from a vascular vantage near cortex                             | Equivalent to a Utah array          |

Two consequences follow and will be repeated in Chapter 2.

First, **decoding is always underdetermined.** Many microscopic arrangements produce the same macroscopic field. A successful decoder has found a usable statistical mapping, not the unique neural code for a proposition.

Second, **cognition is a harder target than reach direction** because the relevant variables are distributed across maps, mixed in single cells, and unstable across contexts. That claim is conceptual, not defeatist. It is the reason Chapter 8 exists.


## 0.14 Plasticity, co-adaptation, and why the user is in the loop

A BCI is not a microphone placed against a fixed transmitter. The nervous system that produces the signal is learning to use the decoder that interprets it. Three timescales matter:

- milliseconds to seconds — synaptic short-term dynamics and fatigue;
- minutes to hours — user strategy and decoder adaptation within a session;
- days to months — skill acquisition and, for implants, tissue response around the electrode.

Cognitive computing, in the narrow sense used in this book, contributes models of the middle timescale: when to slow the interface, when to request a new calibration, when inferred attention no longer supports the current task. Those models are only as good as the measurements in Section 0.13. They do not require a complete theory of mind.


In [ ]:
# Schematic only: a population field as a weighted sum of oscillatory sources.
fs = 256
t = np.arange(0, 2, 1 / fs)
alpha_idle = 1.2 * np.sin(2 * np.pi * 10 * t)
mu_move = 0.3 * np.sin(2 * np.pi * 12 * t)   # attenuated "mu" during imagery
beta = 0.4 * np.sin(2 * np.pi * 20 * t)
noise = 0.35 * np.random.randn(t.size)

eeg_idle = alpha_idle + 0.2 * beta + noise
eeg_imagery = mu_move + 0.2 * beta + noise


def band_power(sig, band, fs=fs, nfft=512):
    spec = np.fft.rfft(sig * np.hanning(sig.size), n=nfft)
    freqs = np.fft.rfftfreq(nfft, 1 / fs)
    mag = (np.abs(spec) ** 2)
    mask = (freqs >= band[0]) & (freqs < band[1])
    return mag[mask].mean()


print(f"8–13 Hz power, idle    = {band_power(eeg_idle, (8, 13)):.3f}")
print(f"8–13 Hz power, imagery = {band_power(eeg_imagery, (8, 13)):.3f}")

fig, axes = plt.subplots(2, 1, figsize=(7.2, 4.4), sharex=True)
axes[0].plot(t, eeg_idle, color="steelblue", linewidth=0.9)
axes[0].set_ylabel("amplitude (a.u.)")
axes[0].set_title("Synthetic scalp trace: idle (stronger 10 Hz)")
axes[1].plot(t, eeg_imagery, color="darkorange", linewidth=0.9)
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("amplitude (a.u.)")
axes[1].set_title("Synthetic scalp trace: motor imagery (attenuated 10–12 Hz)")
plt.tight_layout()
plt.show()


The figure is a cartoon. Real EEG is not a pair of clean sinusoids. The cartoon is enough to fix the idea that a motor-imagery feature can be as simple as “power in a band over a site,” and that the biology underneath is a change in a population rhythm, not the appearance of a new spike code on the scalp.


## Exercises — Part B

**B1.** In two or three sentences, explain why averaging many EEG trials can reveal an event-related potential that is invisible in a single trial, and name one cost of that averaging for a live BCI.

**B2.** A journalist writes that an implant “reads thoughts from a single neuron.” Using Section 0.13, write a four-sentence correction suitable for a technically literate reader.

**B3.** Why might a decoder trained on Monday morning fail on Monday afternoon even if the hardware has not moved? Give one neural reason and one user-strategy reason.

**B4.** Which of the following is a population rhythm, and which is a single-cell event: (i) an action potential; (ii) an 11 Hz mu rhythm over sensorimotor cortex; (iii) a P300 peak in a grand average? One sentence each.

**B5.** A project proposal claims that a consumer EEG headband will “decode working memory contents in real time.” Identify the biological assumption that would have to be true for that claim, and state why Chapter 0 gives you permission to doubt it.


---

## 0.15 What to carry into Chapter 1

Carry four sentences, not the tables.

1. A BCI recording is a linear mixture of fields (and, for implants, of spikes) produced by neural populations; it is not a transcript of a proposition.
2. Linear algebra—inner products, covariance, eigen-directions—is the language in which spatial filtering and linear decoding are stated.
3. Bayes’ rule is the language in which priors, including language models, change a decode.
4. Time-series leakage is the standard way to obtain an impressive number that will not survive contact with a user.

Chapter 1 places these tools inside a definition and a history. You do not need more mathematics or more anatomy to begin.


---

## Further reading (optional, not examinable)

- Strang, G. *Introduction to Linear Algebra*, chapters on orthogonality and eigenvalues.  
- Wasserman, L. *All of Statistics*, chapters on probability and Bayes.  
- Kandel et al., *Principles of Neural Science*, chapters on the neuron and on cortical maps — consult, do not attempt to read cover to cover before Chapter 1.  
- Buzsáki, G. *Rhythms of the Brain*, for a deeper (and opinionated) account of oscillations.

---

*End of Chapter 0.*
